# Lesson 8: Choose units

**You will learn:** how to choose units with a functional localizer, as an fMRI study localizes a region, and compare their lesion with matched random units.
**Requires:** CPU only, about a minute.

`FunctionalSelection` records a region on localizer images, scores each unit by how much more it responds to one category than another (Cohen's d), and keeps the top units. `RandomSelection` draws the same number at random from the same population: the control every localized lesion needs.

**The question:** does silencing the most animal-selective units of the last layer hurt recognition of animals more than recognition of vehicles, and more than silencing random units? The localizer uses the training half; the test uses the other half.

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). It never saw these images.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',
        intervention=build_pytorch_ablation_fn(network))

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(c for c in subject.out_channels if not c.startswith('behavior:'))

['behavior', 'neural:IT', 'neural:V1', 'neural:V4', 'perturbation']

In [5]:
# Half the images (6 per category) fit readouts; the other half test them.
order = stimuli.groupby('category').cumcount()
train = StimulusSet(stimuli[order < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[order >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

In [6]:
from brainscore_core.selection import FunctionalSelection, RandomSelection

K = 2000          # of 86,528 units in features.12 (512 channels x 13 x 13 positions)
animal_units = FunctionalSelection('IT', train, contrast=(['animal'], ['vehicle']),
                                   contrast_column='label', n_units=K).resolve(subject)
population = animal_units.metadata['unit_population']
random_units = RandomSelection(animal_units.layer, n_units=K, n_total=len(population),
                               population=population, seed=0, axis=animal_units.axis,
                               expected_shape=animal_units.expected_shape).resolve(subject)
subject.reset()
print(animal_units.layer, len(animal_units.indices), 'animal-selective units;', len(random_units.indices), 'random units')

features.12 2000 animal-selective units; 2000 random units


In [7]:
from brainscore_core import Perturbation, StateChange, TaskContext
from brainscore_core.responses import Choice
from brainscore_core.streaming import BufferedSession
from brainscore_core.streaming_helpers import apply_state_change

task = TaskContext(response=Choice(['animal', 'vehicle']), fitting_stimuli=train,
                   prefer_path='readout', readout_fit='intact')
labels = np.asarray(test['label'])

def accuracy_by_kind(selection=None):
    applied = None if selection is None else apply_state_change(subject, StateChange(
        kind='ablation', target=selection, perturbation=Perturbation(kind='zero')))
    session = BufferedSession(stimulus_set=test, task_context=task, requested_output_channels=['behavior:choice'])
    subject.participate(session)
    if applied is not None:
        apply_state_change(subject, StateChange(kind='reset', handle_id=applied.handle_id))
    chosen = np.array(['animal', 'vehicle'])[session.collect('behavior:choice').values.argmax(1)]
    return {f'{kind} accuracy': float(np.mean(chosen[labels == kind] == kind)) for kind in ('animal', 'vehicle')}

results = pd.DataFrame({'intact': accuracy_by_kind(), 'animal-selective silenced': accuracy_by_kind(animal_units),
                        'random silenced': accuracy_by_kind(random_units)}).T
results.round(3)

,animal accuracy,vehicle accuracy
intact,0.889,0.750
animal-selective silenced,0.861,0.792
random silenced,0.889,0.750


A localized lesion is informative only next to the random one: if both lower accuracy alike, the selected units were not special. If the animal-selective lesion lowers animal accuracy more than vehicle accuracy, and more than the random lesion does, the localizer found units the readout relies on. With one random draw this is a first look; a study would repeat the random control with several seeds and use held-out localizer images, as here.

## Summary

- `FunctionalSelection` localizes units by contrast; `RandomSelection` matches them.
- Localize on one half of the images, test on the other.

**Next:** [Lesson 9](09_closed_loop.ipynb) runs a subject in a closed loop.